# metallic MAX 相の回帰

学習データにはよくフィットするものの、汎化性能が低く予測能力に乏しいモデルとなるデータの一例


gpt-5にによりコードを生成して修正した。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def get_data(delete_exceptional=True):
    
    df_data =  pd.read_csv("../data/maxphase_property.csv")
    
    target_features = ["Hf (eV/atom)","Ecoh (eV/atom)", "(ρ0 × λ)⊥τ (10^-16 m2)",
     "(ρ0 × λ)⊥λ (10^-16 m2)"]
    
    descriptors = ['n','adirection',
        'bdirection', 'TM_Z', 'TM_row', 'TM_group',
        'TM_atomic_radius_calculated', 'TM_mendeleev_no',
        'TM_electrical_resistivity', 'TM_poissons_ratio', 'TM_molar_volume',
        'TM_thermal_conductivity', 'TM_boiling_point', 'TM_melting_point',
        'TM_liquid_range', 'TM_bulk_modulus', 'TM_youngs_modulus',
        'TM_brinell_hardness', 'TM_rigidity_modulus', 'TM_mineral_hardness',
        'TM_vickers_hardness', 'TM_density_of_solid',
        'TM_coefficient_of_linear_thermal_expansion', 'TM_ionization_energies',
        'TM_atomic_mass', 'TM_atomic_radius', 'TM_average_anionic_radius',
        'TM_average_cationic_radius', 'TM_average_ionic_radius',
        'TM_electron_affinity', 'A_Z', 'A_row', 'A_group',
        'A_atomic_radius_calculated', 'A_mendeleev_no',
        'A_electrical_resistivity', 'A_molar_volume', 'A_thermal_conductivity',
        'A_boiling_point', 'A_melting_point', 'A_liquid_range',
        'A_mineral_hardness', 'A_density_of_solid', 'A_ionization_energies',
        'A_atomic_mass', 'A_atomic_radius', 'A_average_anionic_radius',
        'A_average_cationic_radius', 'A_average_ionic_radius',
        'A_electron_affinity', 'X_Z', 'X_row', 'X_group',
        'X_atomic_radius_calculated', 'X_mendeleev_no', 'X_refractive_index',
        'X_molar_volume', 'X_thermal_conductivity', 'X_boiling_point',
        'X_melting_point', 'X_liquid_range', 'X_ionization_energies',
        'X_atomic_mass', 'X_atomic_radius', 'X_average_anionic_radius',
        'X_average_cationic_radius', 'X_average_ionic_radius',
        'X_electron_affinity',
        'mean_Z',
        'stddev_Z',
        'mean_row',
        'stddev_row',
        'mean_group',
        'stddev_group',
        'mean_atomic_radius_calculated',
        'stddev_atomic_radius_calculated',
        'mean_mendeleev_no',
        'stddev_mendeleev_no',
        'mean_molar_volume',
        'stddev_molar_volume',
        'mean_thermal_conductivity',
        'stddev_thermal_conductivity',
        'mean_boiling_point',
        'stddev_boiling_point',
        'mean_melting_point',
        'stddev_melting_point',
        'mean_liquid_range',
        'stddev_liquid_range',
        'mean_ionization_energies',
        'stddev_ionization_energies',
        'mean_atomic_mass',
        'stddev_atomic_mass',
        'mean_atomic_radius',
        'stddev_atomic_radius',
        'mean_average_anionic_radius',
        'stddev_average_anionic_radius',
        'mean_average_cationic_radius',
        'stddev_average_cationic_radius',
        'mean_average_ionic_radius',
        'stddev_average_ionic_radius',
        'mean_electron_affinity',
        'stddev_electron_affinity']

    if delete_exceptional:
        df_data = df_data[df_data[target_features[2]]<100].reset_index(drop=True)

    return df_data, descriptors, target_features

g_df_data, g_feat_cols, g_target_cols  = get_data(delete_exceptional=True)

In [ ]:
g_df_data

In [ ]:
g_model_result = [] # モデル毎のR2を保存する。

g_target_col = g_target_cols[2] #目的変数
g_target_col

In [ ]:
# 表示ライブラリ

import matplotlib.pyplot as plt

def plot_yy(df_result_all, y1label='y_true', y2label='y_pred_all'):
    fig, ax = plt.subplots(1,1, figsize=(5,5))
    df_result_all.plot.scatter(y1label, y2label,  alpha=0.3, ax=ax)
    
    # 対角線の範囲を決める（両軸の min/max から）
    vmin = min(df_result_all["y_true"].min(), df_result_all["y_pred_all"].min())
    vmax = max(df_result_all["y_true"].max(), df_result_all["y_pred_all"].max())
    
    # 対角線を描く
    ax.plot([vmin, vmax], [vmin, vmax], linestyle="--",)
    
    ax.set_xlabel(y1label)
    ax.set_ylabel(y2label)
    ax.set_aspect("equal")  # 必要なら、1:1 の比率にする

In [ ]:
from sklearn.model_selection import KFold

g_kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
import numpy as np
import pandas as pd

def make_rfr_with_cv_predictions(df_Xy, descriptors, target_feature, cv):
    # --- 入力データ ---
    X = df_Xy[descriptors]
    y = df_Xy[target_feature]

    # --- モデル定義 ---
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    # --- 交差検定設定 ---
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # --- CV 予測値算出 ---
    y_pred_cv = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)

    # --- CV の R² 評価 ---
    r2_scores = []
    for train_idx, test_idx in kf.split(X):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        r2 = model.score(X.iloc[test_idx], y.iloc[test_idx])
        r2_scores.append(r2)

    r2_mean_cv = np.mean(r2_scores)
    print("R²スコア（各分割）:", np.round(r2_scores, 4))
    print("平均R²スコア(CV):", r2_mean_cv)

    # --- 全データで最終モデル学習 ---
    model.fit(X, y)

    # 全データに対する予測と R²
    y_pred_all = model.predict(X)
    r2_all = model.score(X, y)
    print("全データに対する R² スコア:", r2_all)

    # --- 結果まとめ ---
    df_result = df_Xy.copy()
    df_result["y_true"] = y
    df_result["y_pred_cv"] = y_pred_cv
    df_result["y_pred_all"] = y_pred_all

    # モデルに加えて、CV平均R²と全データR²も返す
    return model, df_result, r2_mean_cv, r2_all


# 使用例
model, df_result, r2_mean_cv, r2_all = make_rfr_with_cv_predictions(
    g_df_data, g_feat_cols, g_target_col, g_kf
)


g_model_result.append({'model':'RandomForest','cv_r2': r2_mean_cv,'r2_all': r2_all})

plot_yy(df_result)

# Ridge回帰

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def make_ridge_fit_all_data_with_alpha_cv(
    df_Xy,
    descriptors,
    target_feature,
    alphas=None,
    cv=5
):
    """
    α を交差検証で選択した Ridge 回帰モデルを、
    全データで学習する関数。

    CV の平均 R² も出力・返却する。
    """
    X = df_Xy[descriptors]
    y = df_Xy[target_feature]

    if alphas is None:
        # α の候補（10^-4 〜 10^4 を 20 分割）
        alphas = np.logspace(-4, 4, 20)


    # 標準化 + RidgeCV のパイプライン
    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=alphas, cv=cv, scoring="r2")
    )

    # α 選択を含めて学習（内部で CV）
    model.fit(X, y)

    ridgecv = model.named_steps["ridgecv"]

    # ベストな α を取得
    best_alpha = ridgecv.alpha_
    print("選ばれた alpha:", best_alpha)

    # CV 平均 R² を取得（scoring='r2' のとき）
    cv_r2 = ridgecv.best_score_
    print("CV の平均 R² スコア:", cv_r2)

    # 全データに対する予測
    y_pred_all = model.predict(X)

    # 全データに対する R²
    r2_all = model.score(X, y)
    print("全データに対する R² スコア:", r2_all)

    # 結果まとめ
    df_result = df_Xy.copy()
    df_result["y_true"] = y
    df_result["y_pred_all"] = y_pred_all

    # CV の平均 R² も返す
    return model, df_result, best_alpha, cv_r2, r2_all


# 使用例
model_all, df_result_all, best_alpha, cv_r2, r2_all = make_ridge_fit_all_data_with_alpha_cv(
    g_df_data,
    g_feat_cols,
    g_target_col,
    # 必要なら α の候補を自分で指定
    # alphas=np.logspace(-3, 3, 13),
    cv=g_kf
)

g_model_result.append({'model':'Ridge', 'cv_r2': cv_r2, 'r2_all': r2_all})

plot_yy(df_result_all)

# KernelRidge


In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd


def make_kernelridge_fit_all_data_with_cv(
    df_Xy,
    descriptors,
    target_feature,
    alphas=None,
    gammas=None,
    cv=5,
    scoring="r2"
):
    """
    KernelRidge(RBF) の alpha, gamma を交差検証で最適化し、
    ベストのハイパーパラメータで全データを学習する。

    Parameters
    ----------
    df_Xy : pd.DataFrame
        特徴量と目的変数を含む DataFrame
    descriptors : list[str]
        説明変数として使う列名
    target_feature : str
        目的変数の列名
    alphas : list[float] or np.ndarray, optional
        alpha の候補（None のときは logspace で自動設定）
    gammas : list[float] or np.ndarray, optional
        gamma の候補（None のときは logspace で自動設定）
    cv : int
        交差検証分割数
    scoring : str
        スコアリング指標（デフォルト: "r2"）

    Returns
    -------
    best_model : Pipeline
        StandardScaler + KernelRidge の学習済みモデル
    df_result : pd.DataFrame
        元の df に y_true, y_pred_all を追加したもの
    best_params : dict
        最適な alpha, gamma などのパラメータ
    """

    # --- 入力データ ---
    X = df_Xy[descriptors]
    y = df_Xy[target_feature]

    # デフォルトの探索範囲（必要に応じて狭めてOK）
    if alphas is None:
        alphas = np.logspace(-4, 2, 7)   # 1e-4 ～ 1e2
    if gammas is None:
        gammas = np.logspace(-3, 1, 7)   # 1e-3 ～ 1e1

    # --- モデル（標準化 + KernelRidge） ---
    base_model = make_pipeline(
        StandardScaler(),
        KernelRidge(kernel="rbf")
    )

    # パラメータ名は "ステップ名__パラメータ" 形式
    param_grid = {
        "kernelridge__alpha": alphas,
        "kernelridge__gamma": gammas,
    }

    # --- グリッドサーチ (CV) ---
    
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    grid.fit(X, y)

    # ベストモデル & ベストパラメータ
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_

    print("CV で選ばれたパラメータ:", best_params)
    print("CV ベストスコア (mean {}): {:.4f}".format(scoring, best_score))

    # --- 全データでの予測 ---
    y_pred_all = best_model.predict(X)
    r2_all = best_model.score(X, y)
    print("全データに対する R² スコア:", r2_all)

    # --- 結果まとめ ---
    df_result = df_Xy.copy()
    df_result["y_true"] = y
    df_result["y_pred_all"] = y_pred_all

    return best_model, df_result, best_params, best_score, r2_all


best_model, df_result_all, best_params, cv_r2, r2_all = make_kernelridge_fit_all_data_with_cv(
    g_df_data,
    g_feat_cols,
    g_target_col,
    # 必要なら自分で範囲を指定
    # alphas=np.logspace(-3, 1, 9),
    # gammas=np.logspace(-2, 2, 9),
    cv=g_kf
)

print(best_params)
g_model_result.append({'model':'KernelRidge', 'cv_r2': cv_r2, 'r2_all': r2_all})

plot_yy(df_result_all)

In [ ]:
g_df_result = pd.DataFrame(g_model_result)
print(g_target_col)
g_df_result.set_index('model')

## model performance

###  'Hf (eV/atom)'

| model        |    cv_r2 |   r2_all |
| ------------ | -------: | -------: |
| RandomForest | 0.779002 | 0.971382 |
| Ridge        | 0.749893 | 0.881595 |
| KernelRidge  | 0.780166 | 0.939679 |




### 'Ecoh (eV/atom)'

| model        |    cv_r2 |   r2_all |
| ------------ | -------: | -------: |
| RandomForest | 0.953510 | 0.995249 |
| Ridge        | 0.992149 | 0.996166 |
| KernelRidge  | 0.993959 | 0.999779 |


### (ρ0 × λ)⊥τ (10^-16 m2)


| model        |     cv_r2 |   r2_all |
| ------------ | --------: | -------: |
| RandomForest | -0.071458 | 0.901358 |
| Ridge        |  0.126233 | 0.376628 |
| KernelRidge  |  0.131993 | 0.386843 |



### (ρ0 × λ)⊥λ (10^-16 m²)

| model        |     cv_r2 |   r2_all |
| ------------ | --------: | -------: |
| RandomForest | -0.174281 | 0.883231 |
| Ridge        |  0.090435 | 0.264654 |
| KernelRidge  |  0.081233 | 0.343939 |

